In [7]:
import pandas as pd
import numpy as np
import warnings
import sys
import joblib

# CHANGED: Import SVC from the cuml library for GPU acceleration
from cuml.svm import SVC
# Import the necessary scikit-learn components
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score




In [8]:

train_bc = pd.read_parquet('train_bc.parquet')
test_bc = pd.read_parquet('test_bc.parquet')

# Separate features (X) and target (y)
X_train = train_bc.drop('Attack', axis=1)
y_train = train_bc['Attack']
X_test = test_bc.drop('Attack', axis=1)
y_test = test_bc['Attack']

In [9]:



print(f"Train data loaded: {len(X_train)} samples.")
print(f"Test data loaded: {len(X_test)} samples.")
print("-" * 30)

# --- 1. Define Model Pipeline and Hyperparameter Grid for cuML SVC ---

# Define the Pipeline: StandardScaler is CRITICAL for SVM
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC( # CHANGED: Estimator is now cuML SVC
        random_state=42,
        # GPU memory usage is handled by cuml; no n_jobs setting is needed here
    ))
])

# Define the EXPANDED Hyperparameter Grid for SVC:
param_grid = [
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.1, 1, 10]
    },
    {
        'svm__kernel': ['rbf'],
        'svm__C': [0.1, 1, 10],
        'svm__gamma': ['scale', 0.1, 1]
    }
]





Train data loaded: 30000 samples.
Test data loaded: 15000 samples.
------------------------------


In [10]:
# --- 2. Initialize and Run Grid Search ---

total_combinations = 3 + 9 # 12 combinations

# Initialize Grid Search with 5-fold cross-validation
# n_jobs=-1 here will parallelize the cross-validation *folds* across CPU cores,
# while each individual SVC model trains rapidly on the GPU.
grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='f1',
    verbose=0,
    n_jobs=-1
)

print(f"Starting Grid Search across {total_combinations} total parameter combinations for cuML SVC...")
print("⚠️ Training individual models will be accelerated by the GPU.")
grid_search.fit(X_train, y_train)
print("Grid Search complete.")


Starting Grid Search across 12 total parameter combinations for cuML SVC...
⚠️ Training individual models will be accelerated by the GPU.
Grid Search complete.


In [11]:
# --- 3. Evaluation and Model Saving ---

# 3a. Get Best Model and Params
best_svm = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("\n" + "="*50)
print("✨ CUML SVM HYPERPARAMETER TUNING RESULTS (GPU) ✨")
print(f"Best cross-validation F1 score: {best_score:.4f}")
print(f"Best parameters found: {best_params}")
print("="*50)

# Save the best model
joblib.dump(best_svm, r'binary_svm.joblib')
print(f"Best model saved to models\\binary_svm_gpu.joblib")

# 3b. Evaluate on the Test Set
y_pred = best_svm.predict(X_test)


✨ CUML SVM HYPERPARAMETER TUNING RESULTS (GPU) ✨
Best cross-validation F1 score: 0.9762
Best parameters found: {'svm__C': 10, 'svm__gamma': 1, 'svm__kernel': 'rbf'}
Best model saved to models\binary_svm_gpu.joblib


In [12]:

# Calculate key metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)

print("\n--- FINAL MODEL PERFORMANCE ON TEST SET ---")
print(f"Best Kernel: {best_params.get('svm__kernel', 'N/A')}")
print(f"Test Set Accuracy: {test_accuracy:.4f}")
print(f"Test Set F1-Score: {test_f1:.4f}")

# Print full classification report
print("\nClassification Report (Test Set):\n")
print(classification_report(y_test, y_pred))
print("-" * 50)
print("Script execution finished.")


--- FINAL MODEL PERFORMANCE ON TEST SET ---
Best Kernel: rbf
Test Set Accuracy: 0.9671
Test Set F1-Score: 0.9671

Classification Report (Test Set):

              precision    recall  f1-score   support

           0       0.95      0.99      0.97      7373
           1       0.99      0.95      0.97      7627

    accuracy                           0.97     15000
   macro avg       0.97      0.97      0.97     15000
weighted avg       0.97      0.97      0.97     15000

--------------------------------------------------
Script execution finished.
